# Config

> Where the tunnel goes, what we assume about it, and where files live.


In [ ]:
#| default_exp config

In [ ]:
#| export
import os
from pathlib import Path

import matplotlib as mpl
import numpy as np
import pyproj
from dotenv import load_dotenv
from shapely.geometry import LineString, Point

In [ ]:
from fastcore.test import test_close, test_eq

## Files

Downloads are cached in `data/raw/`, intermediate results go in `data/derived/`, and the blog's charts in `figures/`.


In [ ]:
#| export
ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
RAW, DERIVED, FIGURES = ROOT / "data" / "raw", ROOT / "data" / "derived", ROOT / "figures"
for _d in (RAW, DERIVED, FIGURES):
    _d.mkdir(parents=True, exist_ok=True)

Two inputs need an HM Land Registry account, so we can't download them for you. Say where they are in a `.env` file
at the project root (copy `.env.example`):

- `NPS_DIR`: the National Polygon Service shapefiles (`LR_POLY_FULL_*.shp`), i.e. title plans.
- `COD_DIR`: the CCOD and OCOD files of company owners (`CCOD_FULL_*.zip`, `OCOD_FULL_*.zip`). Defaults to `data/`.


In [ ]:
#| export
load_dotenv(ROOT / ".env")
NPS_DIR = Path(os.environ.get("NPS_DIR", ROOT / "data" / "nps"))
COD_DIR = Path(os.environ.get("COD_DIR", ROOT / "data"))

In [ ]:
test_eq((ROOT / "pyproject.toml").exists(), True)

## Coordinates

Points are given as (longitude, latitude), which is how maps label them. Distances and areas are measured on the
British National Grid (BNG), whose units are metres.


In [ ]:
#| export
BNG, WGS84 = "EPSG:27700", "EPSG:4326"
_to_bng = pyproj.Transformer.from_crs(WGS84, BNG, always_xy=True)


def to_bng(lonlat: tuple[float, float]) -> tuple[float, float]:
    "Convert (lon, lat) to BNG (x, y) metres."
    return _to_bng.transform(*lonlat)

Big Ben is at grid reference TQ 30268 79640:


In [ ]:
test_close(to_bng((-0.12463, 51.50073)), (530_268, 179_640), eps=10)

## The route

The tunnel is an immersed tube on the route TfL sketched in 2017: it crosses the river between the mouth of Nelson
Dock in Rotherhithe and the mouth of the West India Docks impound lock inlet, and ramps up inside each dock to the
street. One Canada Square stands in for "the jobs".

In [ ]:
#| export
SOUTH_RIVER = (536660, 180183)  # mouth of Nelson Dock (BNG, from its OpenStreetMap outline)
SOUTH_HEAD = (536565, 180175)  # head of Nelson Dock, by Rotherhithe Street
NORTH_RIVER = (537036, 180103)  # mouth of the impound lock inlet
NORTH_HEAD = (537111, 180094)  # head of the inlet, by Westferry Road
CANARY_WHARF = (-0.019570, 51.504970)  # One Canada Square (lon, lat)
_to_wgs84 = pyproj.Transformer.from_crs(BNG, WGS84, always_xy=True)


def to_lonlat(xy: tuple[float, float]) -> tuple[float, float]:
    "Convert BNG (x, y) to (lon, lat)."
    return _to_wgs84.transform(*xy)


def river_crossing() -> LineString:
    "The immersed section under the river, dock mouth to dock mouth, in BNG."
    return LineString([SOUTH_RIVER, NORTH_RIVER])

In [ ]:
test_close(river_crossing().length, 384, eps=5)
test_close(to_bng(to_lonlat(SOUTH_RIVER)), SOUTH_RIVER, eps=0.01)

## How deep the tube goes

All heights are in metres above Ordnance Datum Newlyn (ODN). Three published sources set the depth:

- **The quay.** The Environment Agency's flood defence records put the river walls at both ends at 5.2 m to 5.8 m.
  We take 5.5 m.
- **The riverbed.** The Port of London Authority's chart 319 (Lower Pool to Limehouse Reach) gives soundings along
  the crossing. Its tide table puts chart datum 3.35 m below ODN at West India Pier. Below we read the bed off the
  chart every 25 m or so: positive numbers are foreshore that dries at low tide, negative numbers are depths.
- **The tube.** Its roof sits under 1.5 m of cover, with a 1 m thick roof and 3.5 m of headroom beneath that, so the
  walkway is 6 m below the riverbed.

In [ ]:
#| export
QUAY_ODN = 5.5  # EA river wall crests at both ends
CHART_DATUM_ODN = -3.35  # PLA chart 319, West India Pier
COVER_M, ROOF_M, HEADROOM_M = 1.5, 1.0, 3.5
# (metres from the Nelson Dock mouth, riverbed relative to chart datum) read off PLA chart 319
RIVERBED_CD = [
    (0, 2.5), (25, 1.7), (50, 1.1), (75, 0.2), (90, -0.5), (100, -1.0), (125, -2.2), (150, -2.6), (175, -3.6),
    (200, -4.9), (225, -5.6), (250, -5.3), (275, -5.1), (300, -5.0), (325, -1.7), (350, -0.1), (365, 1.3), (384, 2.6),
]  # fmt: skip


def walkway_limit(bed_cd):
    "The highest the walkway can be (m ODN) under a riverbed at `bed_cd` relative to chart datum."
    return CHART_DATUM_ODN + bed_cd - COVER_M - ROOF_M - HEADROOM_M

In [ ]:
test_close(walkway_limit(-5.6), -14.95)

## The ramps

The walkway can't step: it slopes at no more than 1 in 20, the gradient TfL says many UK ramps are built to. So the
highest the walkway can be at any point is set by the deepest limit within reach of it, plus the climb back from
there. That is a lower envelope of V shapes, one per sounding. A tube can't bob up and down between soundings, so we
then make it descend steadily to its lowest point and climb steadily from there.

In [ ]:
#| export
RAMP_GRADIENT = 1 / 20


def walkway_profile(points=RIVERBED_CD, gradient=RAMP_GRADIENT):
    "Distances along the crossing and the walkway level (m ODN) there: as high as the riverbed and gradient allow."
    xs = np.arange(0, points[-1][0] + 1.0)
    limits = [(x, walkway_limit(b)) for x, b in points]
    w = np.array([min(lim + gradient * abs(x - r) for r, lim in limits) for x in xs])
    low = int(w.argmin())  # a tube descends to its lowest point and climbs from there, without dips in between
    w[: low + 1] = np.minimum.accumulate(w[: low + 1])
    w[low:] = np.minimum.accumulate(w[low:][::-1])[::-1]
    return xs, w

In [ ]:
# A deep middle: 100 m of climb at 1 in 20 only gets 5 m back up, so the walkway at the bank is set by the middle
_, w_deep = walkway_profile([(0, 0.0), (100, -10.0), (200, 0.0)], gradient=0.05)
test_close(w_deep[0], walkway_limit(-10.0) + 5)
# A shallow middle: now the bank's own riverbed is what holds the walkway down
_, w_shallow = walkway_profile([(0, 0.0), (100, -2.0), (200, 0.0)], gradient=0.05)
test_close(w_shallow[0], walkway_limit(0.0))
test_close(w_shallow[100], walkway_limit(-2.0))

Under the deep channel the walkway is at its lowest. Towards each bank it climbs, but only as far as the shallow
foreshore allows, so at each river wall it is still well below the quay. The rest of the climb is the ramp on land.

In [ ]:
#| export
def ramp_lengths() -> tuple[float, float]:
    "Length of the ramp on land at the south and north ends."
    _, w = walkway_profile()
    return (QUAY_ODN - w[0]) / RAMP_GRADIENT, (QUAY_ODN - w[-1]) / RAMP_GRADIENT


def tunnel_length_m() -> float:
    "Distance travelled from entrance to entrance: both ramps plus the crossing."
    return sum(ramp_lengths()) + river_crossing().length

In [ ]:
xs, w = walkway_profile()
DEPTH_M = QUAY_ODN - w.min()
{"lowest walkway (m ODN)": round(float(w.min()), 1), "deepest below the quay (m)": round(float(DEPTH_M), 1),
 "ramps on land (m)": [round(r) for r in ramp_lengths()], "entrance to entrance (m)": round(tunnel_length_m())}  # fmt: skip

{'lowest walkway (m ODN)': -14.9,
 'deepest below the quay (m)': 20.4,
 'ramps on land (m)': [247, 313],
 'entrance to entrance (m)': 944}

## Where the ramps go

Neither dock is long enough for a straight ramp: Nelson Dock is about 95 m from mouth to head, the inlet about 75 m.
So each ramp folds back and forth along its dock, in legs as long as the dock, about 9 m apart (a 7 m wide trough
plus walls). The entrance is wherever the last leg ends.

In [ ]:
#| export
RAMP_PITCH_M = 9.0


def folded_ramp(mouth, head, length, pitch=RAMP_PITCH_M) -> LineString:
    "A ramp of `length` starting at `mouth`, folding back and forth along the dock towards `head`, in BNG."
    mouth, head = np.array(mouth, float), np.array(head, float)
    leg = np.linalg.norm(head - mouth)
    along = (head - mouth) / leg
    across = np.array([-along[1], along[0]])
    n = int(np.ceil(length / leg))
    pts, left = [], length
    for k in range(n):
        offset = (k - (n - 1) / 2) * pitch * across
        start, run = (mouth, along) if k % 2 == 0 else (head, -along)
        pts += [start + offset, start + offset + run * min(leg, left)]
        left -= leg
    return LineString(pts)


def ramps() -> tuple[LineString, LineString]:
    "The south and north ramps in plan, each from the river wall to its entrance."
    south, north = ramp_lengths()
    return folded_ramp(SOUTH_RIVER, SOUTH_HEAD, south), folded_ramp(NORTH_RIVER, NORTH_HEAD, north)

In [ ]:
toy_ramp = folded_ramp((0, 0), (100, 0), 250, pitch=10)
test_eq(len(toy_ramp.coords), 6)  # three legs
test_close(toy_ramp.coords[-1], (50, 10))  # the last leg stops halfway, on the third line

Each entrance is the top end of its ramp.

In [ ]:
#| export
def entrances() -> tuple[tuple[float, float], tuple[float, float]]:
    "The south and north entrances, in BNG."
    south, north = ramps()
    return south.coords[-1], north.coords[-1]


SOUTH_PORTAL, NORTH_PORTAL = (to_lonlat(e) for e in entrances())  # (lon, lat), as used by the other notebooks


def tunnel_line() -> LineString:
    "The whole route in plan, entrance to entrance, in BNG."
    south, north = ramps()
    return LineString([*reversed(south.coords), *north.coords])

In [ ]:
{side: [round(c) for c in e] for side, e in zip(("south", "north"), entrances())}

{'south': [536605, 180169], 'north': [537049, 180120]}

The ramps need more room than the docks give them: three legs side by side in Nelson Dock, which is about 24 m wide,
and five in the inlet, which is about 23 m wide. So the folds would spill onto the quays either side, the north ramp
by about 20 m.

## Travel assumptions


In [ ]:
#| export
WALK_KMH = 4.8
CYCLE_KMH = 15.0
LIFT_DELAY_MIN = 1.5  # waiting for and riding a lift, e.g. at the Greenwich foot tunnel

## Study area

Street networks and house sales cover Rotherhithe, Bermondsey, Deptford, the Isle of Dogs and Wapping. The
land readjustment area is a circle around the south portal.


In [ ]:
#| export
STUDY_BBOX = (-0.085, 51.470, 0.012, 51.522)  # (west, south, east, north)
CATCHMENT_M = 750


def catchment() -> Point:
    "The readjustment circle around the south portal, in BNG. Notebook 02 trims it to the Rotherhithe bank."
    return Point(to_bng(SOUTH_PORTAL)).buffer(CATCHMENT_M)

In [ ]:
test_close(catchment().area / 1e4, 3.1416 * 0.75**2 * 100, eps=1)  # ~177 ha

## Money

All money is in 2025 prices.


In [ ]:
#| export
PRICE_YEAR = 2025

## Charts

One palette and style for every figure in the blog.


In [ ]:
#| export
COLOURS = {
    "tunnel": "#e8673a", "southwark": "#2b78d6", "housing": "#ee6a37", "other": "#1db07f", "rest": "#b8b8b0", "private": "#dcdcd6",
    "water": "#d6e4ef", "land": "#f0efea", "park": "#cfe3c8", "jubilee": "#7b868c", "text": "#555555",
}  # fmt: skip
BACKGROUND = "#fbfbf9"


def use_style():
    "Apply the blog's chart style to matplotlib."
    mpl.rcParams.update({
        "figure.facecolor": BACKGROUND, "axes.facecolor": BACKGROUND, "savefig.facecolor": BACKGROUND,
        "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 11, "legend.frameon": False,
        "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#aaaaaa",
        "axes.labelcolor": COLOURS["text"], "xtick.color": COLOURS["text"], "ytick.color": COLOURS["text"],
    })  # fmt: skip